# Criação estruturada de QA para avaliação de RAG

Este notebook monta uma base com 9 exemplos para avaliação:
- 3 perguntas `single-hop`
- 3 perguntas `multi-hop`
- 3 perguntas `globais`

Cada linha contém:
- pergunta
- trechos retirados dos documentos (`trecho_1`, `trecho_2`)
- resposta esperada
- metadados de fonte

Ao final, o dataset é salvo em CSV.

In [8]:
from __future__ import annotations

from pathlib import Path

import pandas as pd

In [9]:
# Resolve automaticamente a raiz do projeto (rodando da raiz ou de notebooks/)
_here = Path.cwd()
if (_here / "pdfs_txt").is_dir():
    PROJECT_ROOT = _here.resolve()
elif (_here.parent / "pdfs_txt").is_dir():
    PROJECT_ROOT = _here.parent.resolve()
else:
    raise FileNotFoundError(f"Não encontrei a pasta 'pdfs_txt' no cwd atual: {_here}")

DOCS_DIR = (PROJECT_ROOT / "pdfs_txt").resolve()
OUTPUT_DIR = (PROJECT_ROOT / "docs").resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

txt_files = sorted(DOCS_DIR.glob("*.txt"))
if not txt_files:
    raise FileNotFoundError(f"Nenhum .txt encontrado em: {DOCS_DIR}")

docs = {p.name: p.read_text(encoding="utf-8", errors="ignore") for p in txt_files}
print("PROJECT_ROOT:", PROJECT_ROOT)
print("Arquivos carregados:", len(docs))
print("Exemplos:", ", ".join(list(docs.keys())[:3]))

PROJECT_ROOT: C:\Users\ADM\Documents\benchmarking-graphrag
Arquivos carregados: 5
Exemplos: phmsa_20260075.txt, phmsa_20260076.txt, phmsa_20260078.txt


In [10]:
# Gabarito curado manualmente a partir dos .txt em pdfs_txt/ (sem busca automática).


In [11]:
qa_rows: list[dict] = [
    {
        "id": "Q01",
        "tipo_pergunta": "single-hop",
        "pergunta": "Qual é o número OPID atribuído à operadora ENBRIDGEHOLDINGS (GRAY OAK) LLC neste relatório?",
        "trecho_1": (
            "A1. Operator’s OPS-issued Operator Identification Number (OPID): / / / / / / 40668 "
            "A2. Name of Operator: _auto-populated based on OPID_ ENBRIDGEHOLDINGS (GRAYOAK) LLC "
            "A3. Address of Operator: 915 N ELDRIDGE PKWY, SUITE 1100"
        ),
        "fonte_trecho_1": "phmsa_20260075.txt",
        "trecho_2": "",
        "fonte_trecho_2": "",
        "resposta_esperada": "40668.",
        "observacoes": "Campo A1; mesmo arquivo que o chunking usará como fonte.",
    },
    {
        "id": "Q02",
        "tipo_pergunta": "single-hop",
        "pergunta": "Em qual estado (State) está o endereço da operadora BRIDGER PIPELINE LLC?",
        "trecho_1": (
            "A1. Operator’s OPS-issued Operator Identification Number (OPID): / / / / / / 31878 "
            "A2. Name of Operator: _auto-populated based on OPID_ BRIDGER PIPELINELLC "
            "A3. Address of Operator: 455 N. POPLAR ST. ... A3c. State: ... WY"
        ),
        "fonte_trecho_1": "phmsa_20260076.txt",
        "trecho_2": "",
        "fonte_trecho_2": "",
        "resposta_esperada": "WY (Wyoming).",
        "observacoes": "",
    },
    {
        "id": "Q03",
        "tipo_pergunta": "single-hop",
        "pergunta": "Qual é o CEP (Zip Code) informado para a ULTRAMAR INC.?",
        "trecho_1": (
            "A1. Operator’s OPS-issued Operator Identification Number (OPID): / / / / / / 20202 ULTRAMAR "
            "A2. Name of Operator: ... INC. A3. Address of Operator: 1 VALERO WAY ... "
            "A3d. Zip Code: ... 78249"
        ),
        "fonte_trecho_1": "phmsa_20260078.txt",
        "trecho_2": "",
        "fonte_trecho_2": "",
        "resposta_esperada": "78249.",
        "observacoes": "",
    },
    {
        "id": "Q04",
        "tipo_pergunta": "multi-hop",
        "pergunta": (
            "No relatório da Enbridge (mesmo arquivo), quais números aparecem em “Additional NRC Report numbers” "
            "(campo A21c) e qual é pelo menos um horário local associado ao relatório ao National Response Center em A21a?"
        ),
        "trecho_1": (
            "|A21a... Local time ... initial operator report to the National Response Center:<br>"
            "2/27/2026 8:35<br>2/27/2026 8:35<br>2/27/2026 14:02|"
        ),
        "fonte_trecho_1": "phmsa_20260075.txt",
        "trecho_2": "A21c. Additional NRC Report numbers submitted by the operator: 1456092 20260075",
        "fonte_trecho_2": "phmsa_20260075.txt",
        "resposta_esperada": (
            "Números adicionais do NRC: 1456092 e 20260075. "
            "Horários em A21a incluem, entre outros, 2/27/2026 8:35 e 2/27/2026 14:02."
        ),
        "observacoes": "Exige combinar tabela A21a com linha A21c.",
    },
    {
        "id": "Q05",
        "tipo_pergunta": "multi-hop",
        "pergunta": (
            "Compare dois acidentes distintos: qual é a “Report Date” no formulário PHMSA da BRIDGER PIPELINE LLC "
            "e qual é a “Report Date” no da EXXONMOBIL PIPELINE COMPANY LLC?"
        ),
        "trecho_1": (
            "**ACCIDENT REPORT – HAZARDOUS LIQUID AND** Report Date 3/26/2026 U.S. Department of Transportation ... "
            "BRIDGER PIPELINELLC"
        ),
        "fonte_trecho_1": "phmsa_20260076.txt",
        "trecho_2": (
            "**ACCIDENT REPORT – HAZARDOUS LIQUID AND** Report Date 3/30/2026 ... "
            "EXXONMOBILPIPELINECOMPANY LLC"
        ),
        "fonte_trecho_2": "phmsa_20260081.txt",
        "resposta_esperada": "Bridger: 3/26/2026. ExxonMobil: 3/30/2026.",
        "observacoes": "Dois arquivos diferentes.",
    },
    {
        "id": "Q06",
        "tipo_pergunta": "multi-hop",
        "pergunta": (
            "No incidente da ULTRAMAR INC., qual foi a indicação inicial de falha (A12) e o que o trecho indica sobre "
            "notificação ao National Response Center (NRC) em A21a?"
        ),
        "trecho_1": (
            "A12. ... SCADA-BASED INFORMATION (SUCH AS ALARM(S), ALERT(S),EVENT(S), AND/OR VOLUME OR PACK CALCULATIONS)"
        ),
        "fonte_trecho_1": "phmsa_20260078.txt",
        "trecho_2": (
            "A21a. ... initial operator report to the National Response Center:<br>2/28/2026 11:35<br>"
            "2/28/2026 10:30 ... NRC NOTIFICATION NOT REQUIRED"
        ),
        "fonte_trecho_2": "phmsa_20260078.txt",
        "resposta_esperada": (
            "Indicação inicial: informação baseada em SCADA (alarmes, alertas, eventos e/ou cálculos de volume/pack). "
            "Para o NRC: horários 2/28/2026 11:35 e 10:30 constam e o registro indica “NRC NOTIFICATION NOT REQUIRED”."
        ),
        "observacoes": "",
    },
    {
        "id": "Q07",
        "tipo_pergunta": "global",
        "pergunta": "Qual é o número OMB (OMB NO) exibido no cabeçalho dos relatórios desta pasta?",
        "trecho_1": "OMB NO: 2137-0047 EXPIRATION DATE 3/31/2024",
        "fonte_trecho_1": "phmsa_20260075.txt",
        "trecho_2": "OMB NO: 2137-0047 EXPIRATION DATE 3/31/2024",
        "fonte_trecho_2": "phmsa_20260081.txt",
        "resposta_esperada": "2137-0047 (repetido em todos os formulários da pasta).",
        "observacoes": "Pergunta de síntese em todo o corpus.",
    },
    {
        "id": "Q08",
        "tipo_pergunta": "global",
        "pergunta": (
            "Segundo o aviso de conformidade com 49 CFR Part 195 no início dos relatórios, "
            "qual o limite de multa civil por violação por dia e qual o teto máximo total mencionado em lei?"
        ),
        "trecho_1": (
            "NOTICE: This report is required by 49 CFR Part 195. Failure to report can result in a civil penalty "
            "not to exceed $100,000 for each violation for each day that such violation persists except that the "
            "maximum civil penalty shall not exceed $1,000,000 as provided in 49 USC 60122."
        ),
        "fonte_trecho_1": "phmsa_20260079.txt",
        "trecho_2": (
            "NOTICE: This report is required by 49 CFR Part 195. Failure to report can result in a civil penalty "
            "not to exceed $100,000 for each violation for each day ... maximum civil penalty shall not exceed "
            "$1,000,000 as provided in 49 USC 60122."
        ),
        "fonte_trecho_2": "phmsa_20260076.txt",
        "resposta_esperada": (
            "Até US$ 100.000 por violação por dia enquanto persistir; teto máximo de US$ 1.000.000, "
            "conforme 49 USC 60122."
        ),
        "observacoes": "Texto legal repetido no topo de cada arquivo.",
    },
    {
        "id": "Q09",
        "tipo_pergunta": "global",
        "pergunta": (
            "De acordo com o bloco sobre Paperwork Reduction Act, qual é a estimativa de horas por resposta "
            "e para onde enviar comentários sobre o fardo da coleta?"
        ),
        "trecho_1": (
            "The OMB Control Number for this information collection is 2137-0047. Public reporting for this collection "
            "of information is estimated to be approximately 12 hours per response, including the time for reviewing "
            "instructions, gathering the data needed, and completing and reviewing the collection of information. "
            "All responses to this collection of information are mandatory. Send comments regarding this burden estimate "
            "or any other aspect of this collection of information ... to: Information Collection Clearance Officer, PHMSA, "
            "Office of Pipeline Safety (PHP-30) 1200 New Jersey Avenue, SE, Washington, D.C. 20590."
        ),
        "fonte_trecho_1": "phmsa_20260075.txt",
        "trecho_2": (
            "_**Important:** Please read the separate instructions ... obtain one from the PHMSA Pipeline Safety Community "
            "Web Page at_ https://www.phmsa.dot.gov/pipeline/library/forms _._"
        ),
        "fonte_trecho_2": "phmsa_20260075.txt",
        "resposta_esperada": (
            "Estimativa de cerca de 12 horas por resposta. Comentários sobre o fardo podem ser enviados ao "
            "Information Collection Clearance Officer, PHMSA, Office of Pipeline Safety (PHP-30), "
            "1200 New Jersey Avenue, SE, Washington, D.C. 20590."
        ),
        "observacoes": "Q09 mistura estimativa + endereço; o trecho 2 reforça onde obter instruções.",
    },
]

qa_df = pd.DataFrame(qa_rows)
qa_df

,id,tipo_pergunta,pergunta,trecho_1,fonte_trecho_1,trecho_2,fonte_trecho_2,resposta_esperada,observacoes
0,Q01,single-hop,Qual é o número OPID atribuído à operadora ENB...,A1. Operator’s OPS-issued Operator Identificat...,phmsa_20260075.txt,,,40668.,Campo A1; mesmo arquivo que o chunking usará c...
1,Q02,single-hop,Em qual estado (State) está o endereço da oper...,A1. Operator’s OPS-issued Operator Identificat...,phmsa_20260076.txt,,,WY (Wyoming).,
2,Q03,single-hop,Qual é o CEP (Zip Code) informado para a ULTRA...,A1. Operator’s OPS-issued Operator Identificat...,phmsa_20260078.txt,,,78249.,
3,Q04,multi-hop,"No relatório da Enbridge (mesmo arquivo), quai...",|A21a... Local time ... initial operator repor...,phmsa_20260075.txt,A21c. Additional NRC Report numbers submitted ...,phmsa_20260075.txt,Números adicionais do NRC: 1456092 e 20260075....,Exige combinar tabela A21a com linha A21c.
4,Q05,multi-hop,Compare dois acidentes distintos: qual é a “Re...,**ACCIDENT REPORT – HAZARDOUS LIQUID AND** Rep...,phmsa_20260076.txt,**ACCIDENT REPORT – HAZARDOUS LIQUID AND** Rep...,phmsa_20260081.txt,Bridger: 3/26/2026. ExxonMobil: 3/30/2026.,Dois arquivos diferentes.
5,Q06,multi-hop,"No incidente da ULTRAMAR INC., qual foi a indi...",A12. ... SCADA-BASED INFORMATION (SUCH AS ALAR...,phmsa_20260078.txt,A21a. ... initial operator report to the Natio...,phmsa_20260078.txt,Indicação inicial: informação baseada em SCADA...,
6,Q07,global,Qual é o número OMB (OMB NO) exibido no cabeça...,OMB NO: 2137-0047 EXPIRATION DATE 3/31/2024,phmsa_20260075.txt,OMB NO: 2137-0047 EXPIRATION DATE 3/31/2024,phmsa_20260081.txt,2137-0047 (repetido em todos os formulários da...,Pergunta de síntese em todo o corpus.
7,Q08,global,Segundo o aviso de conformidade com 49 CFR Par...,NOTICE: This report is required by 49 CFR Part...,phmsa_20260079.txt,NOTICE: This report is required by 49 CFR Part...,phmsa_20260076.txt,Até US$ 100.000 por violação por dia enquanto ...,Texto legal repetido no topo de cada arquivo.
8,Q09,global,De acordo com o bloco sobre Paperwork Reductio...,The OMB Control Number for this information co...,phmsa_20260075.txt,_**Important:** Please read the separate instr...,phmsa_20260075.txt,Estimativa de cerca de 12 horas por resposta. ...,Q09 mistura estimativa + endereço; o trecho 2 ...


In [12]:
# Edite as colunas 'pergunta' e 'resposta_esperada' e, se necessário, ajuste os trechos.
# Validação rápida das quantidades por tipo:
counts = qa_df["tipo_pergunta"].value_counts().to_dict()
expected = {"single-hop": 3, "multi-hop": 3, "global": 3}
assert counts == expected, f"Contagem inválida por tipo. Esperado {expected}, obtido {counts}"

# Garante que não restaram placeholders antes de exportar.
placeholder_pattern = r"\[TODO\]"
placeholder_mask = qa_df["pergunta"].str.contains(placeholder_pattern, regex=True) | qa_df["resposta_esperada"].str.contains(placeholder_pattern, regex=True)
if placeholder_mask.any():
    print("Ainda existem placeholders [TODO]. Edite antes de exportar.")
    display(qa_df.loc[placeholder_mask, ["id", "tipo_pergunta", "pergunta", "resposta_esperada"]])
else:
    print("Tudo certo para exportar.")

Tudo certo para exportar.


In [13]:
OUTPUT_CSV = OUTPUT_DIR / "rag_eval_qa_estruturado.csv"
qa_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
print(f"CSV salvo em: {OUTPUT_CSV}")
print(f"Total de linhas: {len(qa_df)}")

CSV salvo em: C:\Users\ADM\Documents\benchmarking-graphrag\docs\rag_eval_qa_estruturado.csv
Total de linhas: 9
